# Lab 11 — Lo que tu pirámide recuerda

**Alumno:** Juan Diego Hernandez Zavala  
**Matrícula:** 2209978x  
**Entidad asignada:** Baja California  

> Resuelve las celdas marcadas con `TODO`. Cuando termines, descarga el notebook (`Archivo → Descargar → .ipynb`) y súbelo en la página de la actividad. No borres esta celda.


# Lab — Sesión 11: Lo que tu pirámide recuerda

**Simulación avanzada e implementación de modelos poblacionales y de catástrofe (ACD53)**

Cuatro partes. La pirámide de tu entidad es el objeto central: se le ponen
**fechas de nacimiento** a las barras, se sigue a tres cohortes entre dos
censos, se compara grupo por grupo con la **población estable equivalente** del
lab 10 para localizar el mayor abultamiento y el mayor hueco, se comprueba el
grupo 0-4 contra los nacimientos registrados, y se calcula el **momentum** por
dos rutas. La cuarta parte escribe el argumento: el evento, la prueba, la
inercia y el límite.

**Esto va a la Entrega 1**, al bloque de pirámide. El diagnóstico no es la
figura: son los cuatro renglones que la acompañan.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ENTIDAD = "Baja California"   # <- tu entidad
FUENTE  = "INEGI: Censos de Poblacion y Vivienda 1990, 2010 y 2020 (poblacion por edad y sexo); Estadistica de Nacimientos Registrados 2015-2019. Regimen: labs 4, 8 y 10 (fecundidad lab 7; mortalidad femenina CONAPO 2023); saldo migratorio: lab 9"   # <- censos 1990 y 2020 por edad y sexo (INEGI), nacimientos 2015-2020, y tus labs 4, 8, 9 y 10

## Datos y herramientas

1. Tu **pirámide de 2020** por sexo, 17 grupos quinquenales (lab 1/2).
2. Tu **pirámide de 1990** por sexo, los mismos 17 grupos (lab 2). Si no la
   tienes, la parte 1.3 se hace con la de práctica y se declara.
3. Tu **régimen** del lab 10: ₅fₓ y las 17 ₅Mₓ femeninas.
4. Tus **nacimientos registrados 2015-2020** (lab 9).

Las funciones vienen hechas porque ya las construiste: `tabla17()` (lab 10),
`tasa_intrinseca()` (la bisección del lab 10) y `proyectar()` (la proyección
femenina por cohortes de la sesión 10, con un factor sobre la fecundidad para
el escenario de Keyfitz).

Los datos de práctica son los de la **misma entidad de práctica de los labs 9
y 10**. No son datos reales.

In [ ]:
GRUPOS = ['0-4','5-9','10-14','15-19','20-24','25-29','30-34','35-39','40-44','45-49',
          '50-54','55-59','60-64','65-69','70-74','75-79','80+']
EDAD  = np.arange(0, 85, 5)
MEDIO = EDAD + 2.5
FERT  = slice(3, 10)
FRAC_NINAS = 100 / 205
RAIZ = 100_000

# --- datos de PRACTICA (ilustrativos, no reales; la entidad de los labs 9 y 10) -----
MUJ_2020 = np.array([88, 90, 86, 84, 82, 92, 90, 80, 74, 68, 55, 46, 38, 30, 21, 13, 10]) * 1000
HOM_2020 = np.array([92, 92, 89, 85, 86, 100, 97, 83, 76, 68, 54, 43, 36, 27, 18, 10, 7]) * 1000
MUJ_1990 = np.array([92, 84, 76, 68, 60, 52, 45, 38, 32, 26, 20, 16, 13, 10, 7, 5, 4]) * 1000
HOM_1990 = np.array([95, 87, 78, 68, 56, 48, 42, 36, 30, 25, 19, 15, 12, 9, 6, 4, 3]) * 1000
F_PRACTICA  = np.array([0.0850, 0.1250, 0.1000, 0.0700, 0.0320, 0.0070, 0.0005])
MF_PRACTICA = np.array([0.00480, 0.00036, 0.00040, 0.00070, 0.00100, 0.00120, 0.00160, 0.00220, 0.00340, 0.00520,
                        0.0046, 0.0068, 0.0105, 0.0165, 0.0265, 0.0440, 0.1150])
NAC_2015_2020_PRACTICA = 195_000       # nacimientos registrados en el quinquenio (lab 9)


def tabla17(nMx, raiz=RAIZ):
    """Tabla de vida femenina en 17 grupos (lab 10). Devuelve (DataFrame, 5Lx/l0)."""
    m = np.asarray(nMx, dtype=float)
    nqx = 5 * m / (1 + 2.5 * m); nqx[-1] = 1.0
    lx = np.zeros(17); lx[0] = raiz
    for i in range(16):
        lx[i + 1] = lx[i] * (1 - nqx[i])
    ndx = lx * nqx
    nLx = np.zeros(17)
    nLx[:-1] = 5 * lx[1:] + 2.5 * ndx[:-1]
    nLx[-1] = lx[-1] / m[-1]
    Tx = np.cumsum(nLx[::-1])[::-1]
    tab = pd.DataFrame({'grupo': GRUPOS, 'nMx': m, 'nqx': nqx, 'lx': lx, 'ndx': ndx,
                        'nLx': nLx, 'Tx': Tx, 'ex': Tx / lx})
    return tab, nLx / raiz


def tasa_intrinseca(f, L):
    """La r de Lotka por biseccion (lab 10). Devuelve (r, TNR, c_estable, b)."""
    aport = FRAC_NINAS * np.asarray(f) * L[FERT]
    TNR = aport.sum()
    y = lambda r: (np.exp(-r * MEDIO[FERT]) * aport).sum()
    lo, hi = -0.1, 0.1
    for _ in range(100):
        mid = (lo + hi) / 2
        if y(mid) > 1: lo = mid
        else: hi = mid
    r = (lo + hi) / 2
    w = np.exp(-r * MEDIO) * L
    b = 1 / w.sum()
    return r, TNR, b * w, b


def proyectar(N0, f, L, pasos, factor=1.0):
    """Proyeccion femenina cerrada por cohortes, paso de 5 anios (sesion 10).

    factor multiplica la fecundidad desde el primer paso: con factor = 1/TNR la
    fecundidad queda exactamente en el reemplazo (escenario de Keyfitz).
    Devuelve una matriz (pasos+1) x 17 con la poblacion en cada paso.
    """
    N = np.asarray(N0, float).copy(); hist = [N.copy()]
    s_ab = L[16] / (L[15] + L[16])
    for _ in range(pasos):
        Nn = np.zeros(17)
        Nn[1:16] = N[0:15] * L[1:16] / L[0:15]
        Nn[16] = (N[15] + N[16]) * s_ab
        B = 5 * (np.asarray(f) * factor * FRAC_NINAS * (N[FERT] + Nn[FERT]) / 2).sum()
        Nn[0] = B * L[0] / 5
        N = Nn; hist.append(N.copy())
    return np.array(hist)

## Tus datos

Pega tus cuatro listas de población (17 grupos cada una), tu régimen del lab
10 y tus nacimientos del quinquenio. Si dejas alguna lista de 2020 vacía se usan
los de práctica y el notebook lo dice.

In [ ]:
# TODO 0: pega tus datos.
mis_muj_2020 = [131891, 147045, 153042, 154341, 166909, 161994, 150225, 143900, 135279, 126657,
                108599, 83875, 68735, 49722, 34613, 22041, 25440]   # 17 grupos de mujeres, 2020
mis_hom_2020 = [134410, 152609, 159306, 161298, 173450, 166250, 152164, 148169, 138332, 134187,
                112789, 84778, 65287, 44982, 30812, 18833, 18837]   # 17 grupos de hombres, 2020
mis_muj_1990 = [100062, 91063, 89902, 99309, 91646, 75918, 62121, 49179, 36328, 29549,
                23922, 18754, 15617, 11238, 7249, 5021, 5795]   # 17 grupos de mujeres, 1990 (vacia si no la tienes)
mis_hom_1990 = [102769, 94240, 90526, 102807, 92536, 75448, 60562, 48138, 35438, 29052,
                23536, 18425, 15152, 11514, 7171, 4417, 4228]   # 17 grupos de hombres, 1990
mis_f  = [0.0500, 0.1000, 0.1000, 0.0700, 0.0350, 0.0090, 0.0005]         # 7 tasas de fecundidad (lab 7 / lab 10)
mis_M  = [0.001788, 0.000139, 0.000216, 0.000369, 0.000458, 0.000521, 0.000655, 0.000954, 0.001490, 0.002403,
          0.003887, 0.006236, 0.009938, 0.015777, 0.024921, 0.039337, 0.091095]         # 17 tasas de mortalidad femenina (lab 10)
mis_nac_2015_2020 = 304_981   # nacimientos registrados del quinquenio (lab 9)

if len(mis_muj_2020) == 0 or len(mis_hom_2020) == 0 or len(mis_f) == 0 or len(mis_M) == 0:
    MUJ20, HOM20, MUJ90, HOM90 = MUJ_2020, HOM_2020, MUJ_1990, HOM_1990
    f, M, NAC5 = F_PRACTICA, MF_PRACTICA, NAC_2015_2020_PRACTICA
    print("Usando los datos de PRACTICA. Sustituyelos por los de tu entidad.")
else:
    MUJ20, HOM20 = np.asarray(mis_muj_2020, float), np.asarray(mis_hom_2020, float)
    MUJ90 = np.asarray(mis_muj_1990, float) if len(mis_muj_1990) == 17 else MUJ_1990
    HOM90 = np.asarray(mis_hom_1990, float) if len(mis_hom_1990) == 17 else HOM_1990
    if len(mis_muj_1990) != 17: print("Sin piramide 1990 propia: la parte 1.3 usa la de PRACTICA. Declaralo.")
    f, M = np.asarray(mis_f, float), np.asarray(mis_M, float)
    NAC5 = mis_nac_2015_2020 if mis_nac_2015_2020 else NAC_2015_2020_PRACTICA
    print(f"Usando datos propios: {ENTIDAD}")

assert len(MUJ20) == 17 and len(HOM20) == 17 and len(MUJ90) == 17 and len(HOM90) == 17, "Faltan grupos: deben ser 17"
tab, L = tabla17(M)
r, TNR, c_est, b_est = tasa_intrinseca(f, L)
print(f"Regimen: TNR {TNR:.4f} · r intrinseca {1000*r:.2f} por mil · e0 femenina {tab['ex'][0]:.2f}")
print(f"Poblacion 2020: {MUJ20.sum() + HOM20.sum():,.0f} · 1990: {MUJ90.sum() + HOM90.sum():,.0f}")

## Parte 1 — La pirámide, con fechas  *(2.5 puntos)*

Quien tiene edad $x$ a $x+4$ en 2020 nació entre $2016 - x$ y $2020 - x$. Y
en una población cerrada, la cohorte que tenía $x$ años en 1990 tiene $x + 30$
en 2020 y ha sobrevivido en la razón ${}_5L_{x+30} / {}_5L_x$: todo lo que
sobre o falte respecto a eso es migración (o error).

In [ ]:
# TODO 1.1: construye la piramide 2020 por sexo (barh, hombres negativos a la
#   izquierda) y etiqueta cada grupo con su COHORTE DE NACIMIENTO, por ejemplo
#   '25-29 (1991-95)'. Ejes etiquetados, titulo con la entidad, leyenda y fuente.
cohorte = [f"{2016 - x}-{str(2020 - x)[-2:]}" for x in EDAD[:16]] + ["antes de 1940"]
etiquetas = [f"{g} ({c})" for g, c in zip(GRUPOS, cohorte)]
ypos = np.arange(17)
fig, ax = plt.subplots(figsize=(9, 7))
ax.barh(ypos, -HOM20, color='#4878a8', label='Hombres')
ax.barh(ypos, MUJ20, color='#c85450', label='Mujeres')
ax.axvline(0, color='0.3', lw=0.8)
ax.set_yticks(ypos); ax.set_yticklabels(etiquetas, fontsize=8)
lim = max(HOM20.max(), MUJ20.max()) * 1.1
ax.set_xlim(-lim, lim)
ax.set_xticks(ax.get_xticks()); ax.set_xticklabels([f"{abs(int(v)):,}" for v in ax.get_xticks()])
ax.set_xlabel('Poblacion (personas)'); ax.set_ylabel('Grupo de edad (cohorte de nacimiento)')
ax.set_title(f'Piramide de poblacion con cohortes de nacimiento: {ENTIDAD}, 2020')
ax.legend(loc='upper right')
fig.text(0.01, -0.02, f'Fuente: {FUENTE}', fontsize=7, style='italic', wrap=True)
plt.tight_layout(); plt.show()

# TODO 1.2: calcula la RAZON DE SEXOS por grupo (hombres por cada 100 mujeres)
#   para 2020 y presentala en una tabla junto a la cohorte de nacimiento.
#   Identifica EN CODIGO los grupos donde pasa de 105 (mas hombres de los que la
#   razon al nacer y la mortalidad permiten) y di que fenomeno sugiere.
rs = HOM20 / MUJ20 * 100
df_rs = pd.DataFrame({'grupo': GRUPOS, 'cohorte': cohorte, 'hombres': HOM20, 'mujeres': MUJ20,
                      'razon de sexos': rs.round(1)})
display(df_rs)
altos = [g for g, v in zip(GRUPOS, rs) if v > 105]
print(f"Grupos con razon de sexos > 105: {altos}")
print("Solo 45-49 (cohorte 1971-75, razon 105.9) pasa de 105. Despues del nacimiento la mortalidad masculina "
      "es mayor, asi que a esa edad la razon deberia estar por debajo de 103; un exceso de hombres sugiere "
      "INMIGRACION predominantemente masculina (migracion laboral a la frontera, a la maquila y al transito "
      "hacia EE.UU.). En el resto de las edades la razon queda entre 101 y 105, y desde los 60 cae por la "
      "sobremortalidad masculina.")

# TODO 1.3: sigue TRES cohortes entre 1990 y 2020 (ambos sexos): la de 0-4, la
#   de 5-9 y la de 10-14 en 1990, que en 2020 tienen 30-34, 35-39 y 40-44.
#   Para cada una: N1990, N2020, la supervivencia esperada L[x+30]/L[x] (usa la
#   tabla femenina como aproximacion para ambos sexos), el N2020 esperado sin
#   migracion, y el EXCESO en porcentaje. Presentalo en una tabla y di que
#   cohorte recibio (o perdio) mas gente.
filas = []
for i in range(3):
    n90 = MUJ90[i] + HOM90[i]
    n20 = MUJ20[i + 6] + HOM20[i + 6]
    sup = L[i + 6] / L[i]
    esp = n90 * sup
    filas.append({'cohorte': f"{1986 - EDAD[i]}-{str(1990 - EDAD[i])[-2:]}", 'edad 1990': GRUPOS[i],
                  'edad 2020': GRUPOS[i + 6], 'N1990': n90, 'N2020': n20, 'superv. esperada': sup,
                  'N2020 esperado': esp, 'exceso (%)': (n20 / esp - 1) * 100})
df_coh = pd.DataFrame(filas)
display(df_coh.round({'superv. esperada': 4, 'N2020 esperado': 0, 'exceso (%)': 1}))
imax = df_coh['exceso (%)'].idxmax()
print(f"Las tres cohortes tienen entre {df_coh['exceso (%)'].min():.0f}% y {df_coh['exceso (%)'].max():.0f}% MAS gente "
      "de la que dejaria la sola supervivencia: en 30 anios Baja California las recibio por inmigracion.")
print(f"La que mas recibio es la de {df_coh.loc[imax, 'cohorte']} ({df_coh.loc[imax, 'edad 1990']} en 1990, "
      f"{df_coh.loc[imax, 'edad 2020']} en 2020), con {df_coh.loc[imax, 'exceso (%)']:.1f}% de exceso.")

## Parte 2 — Contra su estable equivalente  *(2.5 puntos)*

$$\text{razón}_x = \frac{{}_5c_x^{\text{real}}}{{}_5c_x^{*}}$$

La razón separa dos capas: la **tendencia** (cuánto crecimiento pasado carga
la pirámide) y las **desviaciones** de la tendencia (los eventos).

In [ ]:
# TODO 2.1: calcula la distribucion femenina real c = MUJ20/sum(MUJ20), la razon
#   c / c_est grupo por grupo, y grafica la razon como barras con una linea en 1.
#   Verifica EN CODIGO que c y c_est suman 1.
c_real = MUJ20 / MUJ20.sum()
razon = c_real / c_est
print(f"Suma c real = {c_real.sum():.10f} -> {np.isclose(c_real.sum(), 1)} | "
      f"suma c* estable = {c_est.sum():.10f} -> {np.isclose(c_est.sum(), 1)}")
display(pd.DataFrame({'grupo': GRUPOS, 'cohorte': cohorte, 'c real': c_real, 'c* estable': c_est,
                      'razon c/c*': razon}).round(4))
fig, ax = plt.subplots(figsize=(10, 4.5))
ax.bar(GRUPOS, razon, color='#c85450')
ax.axhline(1, color='black', lw=1.2, ls='--', label='razon = 1 (igual a la estable)')
ax.set_xlabel('Grupo de edad'); ax.set_ylabel('c real / c* estable')
ax.set_title(f'Razon distribucion real / estable, mujeres: {ENTIDAD}, 2020')
ax.legend(); plt.xticks(rotation=45)
fig.text(0.01, -0.04, f'Fuente: {FUENTE}', fontsize=7, style='italic', wrap=True)
plt.tight_layout(); plt.show()

# TODO 2.2: localiza EN CODIGO el grupo con la mayor razon respecto al promedio
#   de sus dos vecinos (el abultamiento) y el de menor razon respecto a sus
#   vecinos (el hueco). Fecha las dos cohortes y propon una causa para cada una,
#   ligada a lo que viste en 1.2 y 1.3.
desv = np.array([razon[i] / ((razon[i - 1] + razon[i + 1]) / 2) for i in range(1, 16)])
i_ab = 1 + desv.argmax()
i_hu = 1 + desv.argmin()
print(f"ABULTAMIENTO: {GRUPOS[i_ab]} (nacidas en {cohorte[i_ab]}), razon {razon[i_ab]:.3f}, "
      f"{(desv[i_ab-1]-1)*100:+.1f}% respecto al promedio de sus vecinos.")
print(f"HUECO:        {GRUPOS[i_hu]} (nacidas en {cohorte[i_hu]}), razon {razon[i_hu]:.3f}, "
      f"{(desv[i_hu-1]-1)*100:+.1f}% respecto al promedio de sus vecinos.")
print(f"\nCausa propuesta del abultamiento ({cohorte[i_ab]}): inmigracion de adultos jovenes. Es la edad de "
      "entrada al mercado laboral y a la universidad, y en 1.3 se vio que las cohortes que hoy tienen 30-44 "
      "recibieron 50-60% mas gente de la que dejaria la supervivencia: Baja California atrae poblacion joven. "
      f"Su razon de sexos ({rs[i_ab]:.1f}) no es masculina, asi que seria migracion de ambos sexos.")
print(f"Causa propuesta del hueco ({cohorte[i_hu]}): no es una falta de nacimientos sino de INMIGRACION. Esa cohorte "
      "tenia 25-29 en 1990, asi que ya habia pasado la edad en que mas se llega a Baja California; las cohortes "
      "mas jovenes (como la de 1971-75, 45-49 hoy, la unica con razon de sexos > 105 en 1.2) absorbieron la "
      "migracion laboral de los ochenta y noventa, y en 1.3 se ve que las cohortes que eran ninas en 1990 "
      "crecieron 50-60%. El hueco es el quiebre entre las cohortes que llegaron jovenes y las que no.")

# TODO 2.3: comprueba el grupo 0-4 contra los nacimientos registrados del
#   quinquenio: esperados = NAC5 * L[0] / 5 (supervivencia del primer grupo),
#   observados = MUJ20[0] + HOM20[0]. Reporta la diferencia en porcentaje y
#   nombra las DOS explicaciones posibles (subconteo censal de menores, salida
#   de familias con ninos) sin decidir todavia cual es.
esp_04 = NAC5 * L[0] / 5
obs_04 = MUJ20[0] + HOM20[0]
dif_04 = (obs_04 / esp_04 - 1) * 100
print(f"\nNacimientos registrados 2015-2019: {NAC5:,}")
print(f"0-4 esperado = NAC5 x L[0]/5 = {esp_04:,.0f} | 0-4 observado en el censo = {obs_04:,.0f}")
print(f"Diferencia: {dif_04:+.1f}%")
print("Dos explicaciones posibles, sin decidir todavia: (1) SUBCONTEO CENSAL de menores de 5 anios, el grupo que "
      "los censos omiten con mas frecuencia; (2) SALIDA DE FAMILIAS CON NINOS (emigracion de menores, incluidos "
      "hijos de madres residentes en BC que viven del otro lado de la frontera).")

## Parte 3 — El momentum, por dos rutas  *(2.5 puntos)*

$$M \approx \frac{b\,e_0}{\sqrt{\text{TNR}}}
\qquad\qquad
M = \frac{N(\infty)}{N(0)} \text{ con la fecundidad fija en el reemplazo (factor } 1/\text{TNR}).$$

La primera es la cota de Keyfitz-Frauenthal con tres números tuyos; la segunda,
la proyección. Se reportan las dos, como rango.

In [ ]:
# TODO 3.1: calcula b como hijas nacidas en 2020 por mujer de la poblacion
#   femenina (sum(f * MUJ20[FERT]) * 0.4878 / sum(MUJ20)), toma e0 de la tabla
#   femenina y la TNR, y calcula la cota M_aprox. Imprime los tres insumos y M.
b_obs = (f * MUJ20[FERT]).sum() * FRAC_NINAS / MUJ20.sum()
e0 = tab['ex'][0]
M_aprox = b_obs * e0 / np.sqrt(TNR)
print(f"b (hijas por mujer al anio) = {b_obs:.5f} ({b_obs*1000:.2f} por mil)")
print(f"e0 femenina = {e0:.2f} anios")
print(f"TNR = {TNR:.4f}")
print(f"M_aprox = b x e0 / sqrt(TNR) = {M_aprox:.4f}")

# TODO 3.2: proyecta la poblacion femenina 200 anios (40 pasos) con factor
#   1/TNR y calcula N(t)/N(0) en t = 25, 50, 100 y 200. Imprimelos. M_proy es el
#   valor final; comprueba EN CODIGO que ya no cambia entre 100 y 200 anios.
hist = proyectar(MUJ20, f, L, 40, factor=1 / TNR)
Nt = hist.sum(axis=1)
ratio_t = Nt / Nt[0]
anios = np.arange(41) * 5
for t in [25, 50, 100, 200]:
    print(f"  N({t})/N(0) = {ratio_t[t // 5]:.4f}")
M_proy = ratio_t[-1]
print(f"M_proy = {M_proy:.4f}")
print(f"¿Estable entre 100 y 200 anios? cambio relativo = {(ratio_t[40] / ratio_t[20] - 1)*100:.3f}% -> "
      f"{np.isclose(ratio_t[40], ratio_t[20], rtol=1e-3)}")

# TODO 3.3: grafica N(t)/N(0) contra el tiempo, con una linea horizontal en 1 y
#   otra en M_aprox, y reporta el momentum como RANGO [min, max] de las dos
#   rutas. Di en una frase cuanto va a crecer tu entidad por pura estructura.
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(2020 + anios, ratio_t, marker='o', ms=3, color='#c85450', label='Proyeccion con fecundidad de reemplazo')
ax.axhline(1, color='black', lw=1, ls='--', label='N(t)/N(0) = 1')
ax.axhline(M_aprox, color='#4878a8', lw=1.2, ls=':', label=f'Cota de Keyfitz-Frauenthal = {M_aprox:.3f}')
ax.set_xlabel('Anio'); ax.set_ylabel('N(t) / N(0), mujeres')
ax.set_title(f'Momentum demografico: {ENTIDAD} (poblacion femenina cerrada)')
ax.legend(); ax.grid(alpha=0.3)
fig.text(0.01, -0.04, f'Fuente: {FUENTE}', fontsize=7, style='italic', wrap=True)
plt.tight_layout(); plt.show()
M_min, M_max = min(M_aprox, M_proy), max(M_aprox, M_proy)
print(f"Momentum: [{M_min:.3f}, {M_max:.3f}]")
print(f"Por pura estructura, aun con la fecundidad fijada hoy en el reemplazo y sin migracion, la poblacion de "
      f"{ENTIDAD} creceria entre {(M_min-1)*100:.0f}% y {(M_max-1)*100:.0f}% antes de estabilizarse.")

## Parte 4 — El argumento, con sus cuatro piezas  *(2.5 puntos)*

In [ ]:
# TODO 4.1 — EL EVENTO
# Elige el abultamiento O el hueco mas claro de tu piramide. Fecha la cohorte
# (anios de nacimiento), describe la marca con sus numeros (razon real/estable,
# razon de sexos, tamanio) y propon UNA causa. Usa la palabra «hipotesis».
print(f"El evento mas claro es el ABULTAMIENTO de {GRUPOS[i_ab]} anios: la cohorte nacida en {cohorte[i_ab]}.")
print(f"Marca: es el grupo mas grande de la piramide ({MUJ20[i_ab] + HOM20[i_ab]:,.0f} personas, "
      f"{MUJ20[i_ab]:,.0f} mujeres), tiene la mayor razon real/estable ({razon[i_ab]:.3f}) y queda "
      f"{(desv[i_ab-1]-1)*100:.1f}% por encima del promedio de sus vecinos; su razon de sexos es {rs[i_ab]:.1f}.")
print("HIPOTESIS: el abultamiento es INMIGRACION de adultos jovenes de ambos sexos llegada en la ultima decada "
      "(estudios y primer empleo en Tijuana, Mexicali y Ensenada), no un pico de nacimientos en 1996-2000.")

In [ ]:
# TODO 4.2 — LA PRUEBA
# Di que dato CONFIRMA o REFUTA tu hipotesis y que salio: la misma cohorte en
# 1990 (parte 1.3), la razon de sexos (1.2), los nacimientos registrados (2.3)
# o el residual por edad del lab 9. Si la prueba no es concluyente, dilo.
# La cohorte 1996-2000 no existia en 1990, asi que la sigo desde el censo 2010
# (INEGI, mismo tabulado del lab 1), cuando tenia 10-14, junto a sus dos vecinas.
TOT_2010 = {'5-9': 303_180, '10-14': 305_472, '15-19': 299_195}   # Censo 2010, ambos sexos
pruebas = []
for g10, i20 in [('5-9', 3), ('10-14', 4), ('15-19', 5)]:
    i10 = GRUPOS.index(g10)
    esp = TOT_2010[g10] * L[i20] / L[i10]
    obs = MUJ20[i20] + HOM20[i20]
    pruebas.append({'cohorte': cohorte[i20], 'edad 2010': g10, 'edad 2020': GRUPOS[i20], 'N2010': TOT_2010[g10],
                    'N2020 esperado': round(esp), 'N2020 observado': obs, 'exceso (%)': round((obs / esp - 1) * 100, 1)})
df_p = pd.DataFrame(pruebas); display(df_p)
print("La prueba CONFIRMA la hipotesis: la cohorte 1996-2000 tenia en 2010 un tamanio igual al de sus vecinas "
      f"(no habia pico de nacimientos), y entre 2010 y 2020 crecio {df_p.loc[1, 'exceso (%)']:.1f}% mas de lo que "
      f"permite la supervivencia, contra {df_p.loc[0, 'exceso (%)']:.1f}% de la cohorte mas joven: el abultamiento "
      "se formo por llegada de gente despues de los 10 anios.")
print(f"La razon de sexos ({rs[i_ab]:.1f}) dice que esa migracion no fue mayoritariamente masculina. Lo que la prueba "
      "no distingue es si es migracion interna o internacional, ni cuanto aporta el error de cobertura entre censos.")

In [ ]:
# TODO 4.3 — LA INERCIA
# Reporta tu momentum como rango y traducelo: cuanto creceria tu entidad si su
# fecundidad se fijara hoy en el reemplazo, en que decadas ocurre ese
# crecimiento, y que le pasa al porcentaje de 60 y mas. Liga esto con tu r
# intrinseca del lab 10: signo de r contra signo de M - 1.
p60 = hist[:, 12:].sum(axis=1) / hist.sum(axis=1) * 100
print(f"Momentum: [{M_min:.3f}, {M_max:.3f}]: con fecundidad de reemplazo desde hoy, la poblacion femenina "
      f"crece {(M_min-1)*100:.0f}-{(M_max-1)*100:.0f}% mas antes de estabilizarse.")
print(f"Ese crecimiento ocurre casi todo en las primeras decadas: ya en 2045 va en {ratio_t[5]:.3f} "
      f"({(ratio_t[5]-1)/(M_proy-1)*100:.0f}% del total) y en 2070 en {ratio_t[10]:.3f}; despues practicamente se detiene.")
print(f"El % de 60 y mas sube de {p60[0]:.1f}% (2020) a {p60[5]:.1f}% en 2045, {p60[10]:.1f}% en 2070 y "
      f"{p60[-1]:.1f}% al final: el crecimiento por inercia viene acompaniado de un envejecimiento fuerte.")
print(f"Signos opuestos: la r intrinseca del lab 10 es NEGATIVA ({1000*r:.2f} por mil) y M - 1 es POSITIVO "
      f"({M_proy-1:+.3f}). El regimen actual, a la larga, hace decrecer a la poblacion; la estructura joven, "
      "a corto plazo, la empuja a crecer. Son dos fuerzas distintas y hoy gana la estructura.")

In [ ]:
# TODO 4.4 — EL LIMITE
# Dos limites, con numero: (a) el momentum supone poblacion CERRADA, y tu saldo
# migratorio del lab 9 dice si la inercia real es mayor o menor; (b) la calidad
# del dato: si tu 0-4 esta por debajo de sus nacimientos, parte de lo que lees
# como historia demografica puede ser el censo. Di cual de los dos te preocupa
# mas para TU entidad.
SM_LAB9, TMN_LAB9 = 244_176, 13.79
print(f"(a) POBLACION CERRADA. El momentum supone que nadie entra ni sale, pero el saldo migratorio neto del lab 9 "
      f"fue de {SM_LAB9:,} personas en 2015-2020 ({TMN_LAB9} por mil al anio, 6.5% de la poblacion de 2020). Como "
      "esa migracion es positiva y llega en edades jovenes y reproductivas, la inercia real es MAYOR que la "
      f"calculada: cada quinquenio vuelve a rejuvenecer la piramide y el {(M_max-1)*100:.0f}% es una cota inferior.")
print(f"\n(b) CALIDAD DEL DATO. El 0-4 censal quedo {abs(dif_04):.1f}% por debajo de lo que dejan los nacimientos "
      "registrados. Si es subconteo, la base de la piramide es mas ancha de lo que se ve y parte de la 'caida de la "
      "natalidad' es el censo; ademas Baja California tiene muchos registros extemporaneos, que inflan los "
      "nacimientos registrados.")
print("\nMe preocupa mas el (a): la migracion mueve la poblacion mas de 13 por mil cada anio y toca todas las "
      "edades jovenes, mientras que el problema de calidad se concentra en un solo grupo (0-4) y apenas cambia "
      "el momentum.")

## Cierre y entrega

**Checklist:**
- [ ] Cada grupo de la pirámide tiene su **cohorte de nacimiento**.
- [ ] La razón de sexos está en tabla y los grupos con más de 105 identificados **en código**.
- [ ] Las tres cohortes 1990 → 2020 tienen su exceso en **porcentaje**.
- [ ] La razón real/estable suma-verificada y graficada con la línea en 1.
- [ ] El 0-4 está comparado con los **nacimientos registrados**.
- [ ] El momentum está reportado como **rango** de dos rutas.
- [ ] En la parte 4 usé la palabra **hipótesis** y dije qué dato la prueba.

**Cómo se entrega:** en Colab, `Archivo → Descargar → Descargar .ipynb`, y
arrastra ese archivo al sistema. **El archivo, no el enlace.** Ejecuta el
notebook completo antes de subirlo.

Tienes **dos intentos**. Guarda este notebook: sus cuatro renglones son el
bloque de pirámide de la Entrega 1, y el martes (sesión 12) se ensambla con los
otros diez labs.